In [1]:
import pandas as pd
import numpy as np

In [2]:
# Chapter 10. Data Aggregation and Group Operations

In [3]:
# pandas provides a versatile .groupby() interface, enabling to slice and summarize datasets in a natural way
# query languages impose certain limitations on the kinds of group operations
# with the Python expressiveness can express as custom Python functions 

In [4]:
# 10.1 How to think About Group Operations

In [5]:
# group operations : split-apply-combine
# Series, DataFrame is split into groups based on one or more keys that you provide
# splitting is performed on a particular axis of an object
# once split, function is applied to each group, producing a new value
# results of all those function applications are combined into a result object

In [6]:
# each grouping key can take many forms, and the keys do not have to be all the same type:
# - A list or array of values that is the same length as the axis being grouped
# - A value indicating a column name in a DataFrame
# - A dictionary or Series giving a correspondence between the values on the axis being grouped and the group names
# - A function to be invoked on the axis index or the individuals labels in the index

In [7]:
df = pd.DataFrame({"key1":["a","a",None,"b","b","a",None],
                  "key2":pd.Series([1,2,1,2,1,None,1],dtype="Int64"),
                  "data1":np.random.standard_normal(7),
                  "data2":np.random.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,0.415135,-0.610794
1,a,2,-0.528216,-1.042031
2,None,1,2.024867,-0.351613
3,b,2,0.715525,-0.966031
4,b,1,2.023383,0.580059
5,a,<NA>,-0.122622,0.124440
6,None,1,0.958752,-0.696364


In [8]:
# suppose you want to compute the mean of the data1 column using the labels from key1

In [9]:
# method 1 : access data1 and call groupby with the column (a Series) at key1:

grouped = df["data1"].groupby(df["key1"])
grouped

In [10]:
# grouped is now a special "GroupBy" object
# to compute group means we can call the GroupBy's .mean() method:
grouped.mean()

# here the data (a Series) has been aggregated by splitting the data on the group key;
# producing a new Series that is now indexed by the unique values in the key1 column
# the result index has the name "key1" because the DataFrame column df["key1"] did

key1
a   -0.078568
b    1.369454
Name: data1, dtype: float64

In [11]:
# if instead passed multiple arrays as a list, we'd get something different:
means = df["data1"].groupby([df["key1"],df["key2"]]).mean()
means

key1  key2
a     1       0.415135
      2      -0.528216
b     1       2.023383
      2       0.715525
Name: data1, dtype: float64

In [12]:
# here we grouped the data using two keys, and the resulting Series now has a heirarchical index;
# which consist of the unique pairs of keys observed:
means.unstack()

key2,1,2
key1,,
a,0.415135,-0.528216
b,2.023383,0.715525


In [13]:
# in this example, the group keys are all Series, though they could be any arrays of the right lenght:
states = np.array(["OH","CA","CA","OH","OH","CA","OH"])
years = [2005,2005,2006,2005,2006,2005,2006]
df["data1"].groupby([states,years]).mean()

CA  2005   -0.325419
    2006    2.024867
OH  2005    0.565330
    2006    1.491067
Name: data1, dtype: float64

In [14]:
# frequently, the grouping information is the same DataFrame as the data you want to work on
# in this caes, you can pass column names as the group keys:
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,-0.078568,-0.509461
b,1.5,1.369454,-0.192986


In [15]:
df.groupby("key2").mean()

# there is no key1 column in the result since df["key1"] is not numeric data
# non-numeric data is said to be a nuisance column which is automatically excluded from result
# by default, all of the numeric columns are aggregated

TypeError: agg function failed [how->mean,dtype->object]

In [16]:
df.groupby(["key1","key2"]).mean()

data1     data2
key1 key2                    
a    1     0.415135 -0.610794
     2    -0.528216 -1.042031
b    1     2.023383  0.580059
     2     0.715525 -0.966031

In [17]:
# regardless of the objective in using .groupby(), generally useful GroupBy method is .size()
# .size() returns a Series containing group sizes:

df.groupby(["key1","key2"]).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

In [18]:
# any missing values in a group key are excluded from the result by default
# this behavior can be disabled by passing dropna=False to .groupby:

df.groupby("key1",dropna=False).size()

key1
a      3
b      2
NaN    2
dtype: int64

In [19]:
df.groupby(["key1","key2"],dropna=False).size()

key1  key2
a     1       1
      2       1
      <NA>    1
b     1       1
      2       1
NaN   1       2
dtype: int64

In [20]:
# a group function similar to .size() is .count(), which computes the number of non-null values in each group:

df.groupby("key1").count()

,key2,data1,data2
key1,,,
a,2,3,3
b,2,2,2


In [21]:
# Iterating over Groups
# the object returned by .groupby() supports iteration, generating a sequence of 2-tuples containing the group name along with the chunk of data

df

,key1,key2,data1,data2
0,a,1,0.415135,-0.610794
1,a,2,-0.528216,-1.042031
2,None,1,2.024867,-0.351613
3,b,2,0.715525,-0.966031
4,b,1,2.023383,0.580059
5,a,<NA>,-0.122622,0.124440
6,None,1,0.958752,-0.696364


In [22]:
for fuck,shit in df.groupby("key1"):
    print(fuck) # print the label, which is indexes under key1
    print(shit) # print the corresponding data to the label

# for loop above uses 2 variables - 1 for label and 1 for the corresponding data
# .groupby() requires 2 variables in for loop, since .groupby yields a tuple containing (Label, Data)

a
  key1  key2     data1     data2
0    a     1  0.415135 -0.610794
1    a     2 -0.528216 -1.042031
5    a  <NA> -0.122622  0.124440
b
  key1  key2     data1     data2
3    b     2  0.715525 -0.966031
4    b     1  2.023383  0.580059


In [23]:
# in the case of multiple keys, the first elemnt in the tuple will be a tuple of key values:
for(k1,k2),group in df.groupby(["key1","key2"]):
    print((k1,k2))
    print(group)

# for loop above : (k1,k2) and group are variables

('a', np.int64(1))
  key1  key2     data1     data2
0    a     1  0.415135 -0.610794
('a', np.int64(2))
  key1  key2     data1     data2
1    a     2 -0.528216 -1.042031
('b', np.int64(1))
  key1  key2     data1     data2
4    b     1  2.023383  0.580059
('b', np.int64(2))
  key1  key2     data1     data2
3    b     2  0.715525 -0.966031


In [24]:
# you can choose to do whatever you want with the pieces of data
# computing a dictionary of the data pieces as a one-liner:
pieces = {name:group for name,group in df.groupby("key1")}
pieces["b"]

,key1,key2,data1,data2
3,b,2,0.715525,-0.966031
4,b,1,2.023383,0.580059


In [25]:
# by default .groupby() groups on axis="index" but you can group on any other of the axes
# grouping the columns of the df above by whether they start with "key" or "data":

grouped = df.groupby({"key1":"key","key2":"key","data1":"data","data2":"data"},axis="columns")
grouped

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11080\3589335613.py:4: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  grouped = df.groupby({"key1":"key","key2":"key","data1":"data","data2":"data"},axis="columns")


In [26]:
# we can print out the above groups like so:

for group_key,group_values in grouped:
    print(group_key)
    print(group_values)

data
      data1     data2
0  0.415135 -0.610794
1 -0.528216 -1.042031
2  2.024867 -0.351613
3  0.715525 -0.966031
4  2.023383  0.580059
5 -0.122622  0.124440
6  0.958752 -0.696364
key
   key1  key2
0     a     1
1     a     2
2  None     1
3     b     2
4     b     1
5     a  <NA>
6  None     1


In [27]:
# Grouping with Dictionaries and Series

# grouping information may exist in a form other than an array

people = pd.DataFrame(np.random.standard_normal((5,5)),
                      columns=["a","b","c","d","e"],
                      index=["Joe","Steve","Wanda","Jill","Trey"])
people

,a,b,c,d,e
Joe,-0.445280,0.257245,-0.180319,0.727968,-1.194167
Steve,-1.081276,-0.566638,0.360222,-0.937367,-1.548708
Wanda,-1.785714,-0.326371,-0.554290,1.150792,-1.417224
Jill,0.067108,1.304970,-2.654658,-0.832096,-0.858222
Trey,0.703127,-1.809346,-0.088287,0.300856,1.510744


In [28]:
people.iloc[2:3,[1,2]]=np.nan # adding a few NaN values
people

,a,b,c,d,e
Joe,-0.445280,0.257245,-0.180319,0.727968,-1.194167
Steve,-1.081276,-0.566638,0.360222,-0.937367,-1.548708
Wanda,-1.785714,NaN,NaN,1.150792,-1.417224
Jill,0.067108,1.304970,-2.654658,-0.832096,-0.858222
Trey,0.703127,-1.809346,-0.088287,0.300856,1.510744


In [29]:
# suppose we get a group correspondence for the coluns and want to sum the columns by group:

mapping = {"a":"red","b":"red","c":"blue","d":"blue","e":"red","f":"orange"}
mapping

{'a': 'red', 'b': 'red', 'c': 'blue', 'd': 'blue', 'e': 'red', 'f': 'orange'}

In [30]:
# now, construct an array from this dictionary to pass to a .groupby()
# but instead we can just pass the dictionary (additional key "f" has been added to show unused grouping keys are okay)

by_column = people.groupby(mapping,axis="columns")
by_column

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11080\94912838.py:4: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  by_column = people.groupby(mapping,axis="columns")


In [31]:
by_column.sum()

# this is like conditional sum. By assiging grouping logic, given in dictionary, data can be grouped and aggregated.

,blue,red
Joe,0.547649,-1.382202
Steve,-0.577144,-3.196622
Wanda,1.150792,-3.202938
Jill,-3.486753,0.513856
Trey,0.212569,0.404525


In [32]:
# the same functionality holds for Series, which can be viewed as a fixed-size mapping:

map_series = pd.Series(mapping)
map_series

a       red
b       red
c      blue
d      blue
e       red
f    orange
dtype: object

In [33]:
people.groupby(map_series,axis="columns").count()

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11080\172532021.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  people.groupby(map_series,axis="columns").count()


,blue,red
Joe,2,3
Steve,2,3
Wanda,1,2
Jill,2,3
Trey,2,3


In [34]:
# Grouping with Functions

# defining functions for group mapping
# any function passed as a group key will be called once per index value with the return values being used as the group names
# suppose you want to group by name length - in this case passing a .len() works:

people.groupby(len).sum()

,a,b,c,d,e
3,-0.445280,0.257245,-0.180319,0.727968,-1.194167
4,0.770234,-0.504375,-2.742945,-0.531239,0.652522
5,-2.866990,-0.566638,0.360222,0.213425,-2.965932


In [39]:
# mixing functions with arrays, dictionaries, or Series is not a problem, as everything can be converted to arrays internally:

key_list = ["one","one","one","two","two"]
key_list

['one', 'one', 'one', 'two', 'two']

In [43]:
people

,a,b,c,d,e
Joe,-0.445280,0.257245,-0.180319,0.727968,-1.194167
Steve,-1.081276,-0.566638,0.360222,-0.937367,-1.548708
Wanda,-1.785714,NaN,NaN,1.150792,-1.417224
Jill,0.067108,1.304970,-2.654658,-0.832096,-0.858222
Trey,0.703127,-1.809346,-0.088287,0.300856,1.510744


In [42]:
people.groupby([len,key_list]).min()

# grouping together: lenght of indexes & given key_list
# grouping results become tuples:
# [Joe,3,one] --> (3,"one")
# [Steve,5,one] --> (5,"one")
# [Wanda,5,one] --> (5,"one")
# [Jill,4,two] --> (4,"two")
# [Trey,4,two] --> (4,"two")

,,a,b,c,d,e
3,one,-0.445280,0.257245,-0.180319,0.727968,-1.194167
4,two,0.067108,-1.809346,-2.654658,-0.832096,-0.858222
5,one,-1.785714,-0.566638,0.360222,-0.937367,-1.548708


In [49]:
# Grouping by Index Levels

# aggregating using one of the levels of an axis index

columns = pd.MultiIndex.from_arrays([["US","US","US","JP","JP"],
                                     [1,3,5,1,3]],
                                    names=["cty","tenor"])
columns

MultiIndex([('US', 1),
            ('US', 3),
            ('US', 5),
            ('JP', 1),
            ('JP', 3)],
           names=['cty', 'tenor'])

In [50]:
hier_df = pd.DataFrame(np.random.standard_normal((4,5)),columns=columns) # note that calling columns it's the object columns, not "columns"
hier_df

cty          US                            JP          
tenor         1         3         5         1         3
0      0.437912 -1.397661 -0.260477 -0.758908  0.546776
1     -0.096490  1.527221 -1.516963  0.078184  0.822456
2      0.829947  1.052738 -0.148076 -0.343581 -1.013595
3     -1.787382 -0.714422 -0.244483  0.742452  0.564792

In [53]:
# to group by level, pass the level number or name using the "level" keyword:

hier_df.groupby(level="cty",axis="columns").count()

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11080\1029121770.py:3: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  hier_df.groupby(level="cty",axis="columns").count()


cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


In [54]:
# 10.2 Data Aggregation

# Aggregations refer to any data transformation that produces scalar values from arrays - mean, count, min, and sum

# Optimized common aggregations groupby methods
# .any(), .all() : Return True if any (one or more values) or all non-NA values are "truthy"
# .count() : Number of non-NA values
# .cummin(), .cummax() : Cumulative minimum and maximum of non-NA values
# .cumsum() : Cumulative sum of non-NA values
# .cumprod() : Cumulative product of non-NA values
# .first(), .last() : First and Last non-NA values
# .mean() : Mean of non-NA values
# .median() : Arithmetic median of non-NA values
# .min(), .max() : Minimum and Maximum of non-NA values
# .nth() : Retrieve value that would appear at position n with the data in sorted order
# .ohlc() : Compute four "open-high-low-close" statistics for time series-like data
# .prod() : Product of non-NA values
# .quantile() : Compute sample quantile
# .rank() : Ordinal ranks of non-NA values, like calling Series.rank()
# .size() : Compute group size, returning result as a Series
# .std(), .var() : Sample standard deviation and variance

In [56]:
# we can also use aggregrations of our own, even though not explicitly implemented for Groupby
# we can use .nsmallets()
# internally, Groupby slices up the Series, calls piece.nsmallest(n) for each piece, and then assembles those results into the result object:

df

,key1,key2,data1,data2
0,a,1,0.415135,-0.610794
1,a,2,-0.528216,-1.042031
2,None,1,2.024867,-0.351613
3,b,2,0.715525,-0.966031
4,b,1,2.023383,0.580059
5,a,<NA>,-0.122622,0.124440
6,None,1,0.958752,-0.696364


In [57]:
grouped = df.groupby("key1")
grouped

In [58]:
grouped["data1"].nsmallest(2) # .nsmallest() returns the first n rows with the smallest values in a specified column

# it can be like .nsmallest() == .sort_values() (Ascending) and head(n) combined together

key1   
a     1   -0.528216
      5   -0.122622
b     3    0.715525
      4    2.023383
Name: data1, dtype: float64

In [59]:
# to use your own aggregation functions, pass any function that aggregates an array to the aggregate method or its short alias agg:

def peak_to_peak(arr):
    return arr.max() - arr.min()

In [60]:
grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,0.943351,1.166471
b,1,1.307858,1.546090


In [61]:
# you may notice that some methods, like .describe() also work, even though they are not aggregations:

grouped.describe()

key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0 -0.078568  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0  1.369454  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     0.146257  0.415135   3.0 -0.509461  0.589801 -1.042031 -0.826412   
b     1.696418  2.023383   2.0 -0.192986  1.093251 -0.966031 -0.579509   

                                    
           50%       75%       max  
key1                                
a    -0.610794 -0.243177  0.124440  
b    -0.192986  0.193536  0.580059  

[2 rows x 24 columns]

In [62]:
# Note : Custom aggregation functions are generally much slower than the optimized functions 
#        this is because there is some extra overhead (function cells, data rearrangement) in constructing the intermediate group data chunks

In [63]:
# Column-Wise and Multiple Function Application

tips = pd.read_csv("examples/tips.csv")
tips.head()

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


In [64]:
# add a tip_pct column with the tipe percentage of the total bill:

tips["tip_pct"] = tips["tip"]/tips["total_bill"]
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [65]:
# aggregating a Series or all of the columns of a DataFrame is a matter of using agg with desired function
# however, you may want to agg using a different function, depending on the column, or multiple functions at once
# first, group the tips by day and smoker:

grouped = tips.groupby(["day","smoker"])

In [70]:
# Note : for descriptive statistics, you can pass the name of the function as a string, within .agg()

grouped_pct = grouped["tip_pct"]

In [68]:
grouped_pct.agg("mean")

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

In [69]:
# if you pass a list of functions or function names instead, you get back a DataFrame with column names taken from the functions:

grouped_pct.agg(["mean","std",peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

In [72]:
# you don't need to accept the names that GroupBy gives to the columns; notably, lambda functions have the name "<lambda>", which is hard to identify
# if you pass a list of (name,function) tuples, the first element of each tuple will be used as the DataFrame column names
# in tuple format .agg([("name","function"),("name","function")])

grouped_pct.agg([("average","mean"),("stdev",np.std)])

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11080\1529413272.py:5: FutureWarning: The provided callable <function std at 0x00000177236E00E0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  grouped_pct.agg([("average","mean"),("stdev",np.std)])


average     stdev
day  smoker                    
Fri  No      0.151650  0.028123
     Yes     0.174783  0.051293
Sat  No      0.158048  0.039767
     Yes     0.147906  0.061375
Sun  No      0.160113  0.042347
     Yes     0.187250  0.154134
Thur No      0.160298  0.038774
     Yes     0.163863  0.039389

In [73]:
# with a DataFrame you have more options, as you can specify a list of functions to apply to all of the columns or different functions per column
# suppose we wanted to compute the same three statistics for the tip_pct and total_bill columns:

functions = ["count","mean","max"]

In [74]:
result = grouped[["tip_pct","total_bill"]].agg(functions)
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

In [75]:
# the result DataFrame has hierarchical columns, the same you would get aggregating each column separately and using concat to glue the results together using the column names as the keys argument:
result["tip_pct"]

count      mean       max
day  smoker                           
Fri  No          4  0.151650  0.187735
     Yes        15  0.174783  0.263480
Sat  No         45  0.158048  0.291990
     Yes        42  0.147906  0.325733
Sun  No         57  0.160113  0.252672
     Yes        19  0.187250  0.710345
Thur No         45  0.160298  0.266312
     Yes        17  0.163863  0.241255

In [76]:
# a list of tuples with custom names can be passed:

ftuples = [("average","mean"),("Variance",np.var)]

In [77]:
grouped[["tip_pct","total_bill"]].agg(ftuples)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11080\2321675487.py:1: FutureWarning: The provided callable <function var at 0x00000177236E0220> is currently using SeriesGroupBy.var. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "var" instead.
  grouped[["tip_pct","total_bill"]].agg(ftuples)


tip_pct           total_bill            
              average  Variance    average    Variance
day  smoker                                           
Fri  No      0.151650  0.000791  18.420000   25.596333
     Yes     0.174783  0.002631  16.813333   82.562438
Sat  No      0.158048  0.001581  19.661778   79.908965
     Yes     0.147906  0.003767  21.276667  101.387535
Sun  No      0.160113  0.001793  20.506667   66.099980
     Yes     0.187250  0.023757  24.120000  109.046044
Thur No      0.160298  0.001503  17.113111   59.625081
     Yes     0.163863  0.001551  19.190588   69.808518

In [78]:
# suppose you wanted to apply potentially diferent funcitons to one or more of the columns
# pass a dictionary to .agg() that contains a mapping of column names to any of the function specifications listed so far:

grouped.agg({"tip":np.max,"size":"sum"})

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11080\2149908033.py:4: FutureWarning: The provided callable <function max at 0x00000177236D3560> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped.agg({"tip":np.max,"size":"sum"})


tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [79]:
grouped.agg({"tip_pct":["min","max","mean","std"],
             "size":"sum"})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

In [80]:
# a DataFrame will have hierarchical columns only if multiple funcitons are applied to at least one column

In [82]:
# Returning Aggregated Data Without Row Indexes

# you can disable the aggregated data coming with an index, in most cases, by passing "as_index=False" to .groupby():

tips

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...
239,29.03,5.92,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,No,Sat,Dinner,2,0.098204


In [84]:
tips.groupby(["day","smoker"],as_index=False).mean(numeric_only=True)

# ** Pandas has updated and become more rigid. In older version, it would automatically drop text (string) columns
#     but with update, the automatic text drop process is gone, and need to explicitly specify "numeric_only=True"

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863
